In [1]:
import requests
from utils import split_image, text_to_img, pack_bitmap, convert_to_printer_image
from PIL import Image

esp_ip = "10.76.7.129"

In [ ]:
# english
response = requests.post(
    f"http://{esp_ip}/printer/text",
    json={
        "text": "A",
        "font": "A",
        "bold": True,
        "underline": True,
        "invert": False,
        "width": 1,
        "height": 1,
        "align": "center",
        "feedAfter": 2,
    },
    timeout=30,
)

print(response.status_code)
print(response.text)

200
{"success":true,"message":"Formatted text sent to printer"}


In [ ]:
# chinese
text = "你好呀！\n我是打印机~\n".encode("gb2312")

response = requests.post(
    f"http://{esp_ip}/printer/text",
    params={
        "font": "B",
        "bold": 1,
        "underline": 0,
        "invert": 0,
        "width": 1,
        "height": 1,
        "align": "center",
        "feedAfter": 3,
        "chinese": 1,
    },
    data=text,
    headers={
        "Content-Type": "application/octet-stream"
    },
    timeout=30,
)

print(response.status_code)
print(response.text)

200
{"success":true,"message":"Encoded text sent to printer"}


In [47]:
# Feed three blank lines
response = requests.post(
    f"http://{esp_ip}/printer/feed",
    params={"lines": 3},
    timeout=10,
)
print(response.json())

{'success': True, 'message': 'Paper feed sent to printer'}


In [ ]:
# Run printer test
response = requests.get(
    f"http://{esp_ip}/printer/test",
    timeout=10,
)
print(response.json())

In [ ]:
# print text as image

img = text_to_img(
    text="你好呀！\nHello!\n我是打印机~\n",
    font_path="C:\Windows\Fonts\msyh.ttc",
    font_size=48,
    rotate_180=True,
)

img.save("preview.png")

bitmap = pack_bitmap(img)

response = requests.post(
    f"http://{esp_ip}/printer/image",
    params={
        "width": img.width,
        "height": img.height,
    },
    data=bitmap,
    headers={
        "Content-Type": "application/octet-stream",
    },
    timeout=30,
)

print(response.status_code)
print(response.json())

In [62]:
# print image

with Image.open("./pictures/photo.jpg") as source:
    printer_image = convert_to_printer_image(
        source,
        pixel_size=10,
        contrast=1.2,
        brightness=1.0,
        dither=True,
        rotate_180=True,
    )

printer_image.save("./pictures/preview.png")

In [10]:
source = Image.open("./pictures/test_pic.jpg")

# Only black and white.
result = convert_to_printer_image(
    source,
    grayscale_levels=4,
    dither=True,
    pixel_size=6
)

result.save("./pictures/preview.png")

In [ ]:
# split image if it is longer than 1200
img = Image.open("preview.png").convert("1")

chunks = split_image(img, 200)

for i, chunk in enumerate(chunks):
    chunk.save(f"chunk_{i}.png")
    bitmap = pack_bitmap(chunk)

    response = requests.post(
        f"http://{esp_ip}/printer/image",
        params={
            "width": chunk.width,
            "height": chunk.height,
        },
        data=bitmap,
        headers={
            "Content-Type": "application/octet-stream",
        },
        timeout=30,
    )

    print(response.status_code)
    print(response.json())

200
{'success': True, 'message': 'Image sent to printer'}
200
{'success': True, 'message': 'Image sent to printer'}


In [63]:
bitmap = pack_bitmap(printer_image)

response = requests.post(
    f"http://{esp_ip}/printer/image",
    params={
        "width": printer_image.width,
        "height": printer_image.height,
    },
    data=bitmap,
    headers={
        "Content-Type": "application/octet-stream",
    },
    timeout=30,
)

print(response.status_code)
print(response.json())

ConnectionError: ('Connection aborted.', ConnectionResetError(10054, '远程主机强迫关闭了一个现有的连接。', None, 10054, None))

In [64]:
from convert import convert_image

In [67]:
bytes, width, height = convert_image("./pictures/photo.jpg")
response = requests.post(
    f"http://{esp_ip}/printer/image",
    params={
        "width": width,
        "height": height,
    },
    data=bytes,
    headers={
        "Content-Type": "application/octet-stream",
    },
    timeout=30,
)

print(response.status_code)
print(response.json())

200
{'success': True, 'message': 'Image sent to printer'}
